# Week 9 Mini Project: Build Your First Neural Network

**Project:** Iris Flower Classification  
**Estimated time: 35 minutes**

In this mini project, you will train a small neural network to identify an iris flower as one of three species:

- Setosa
- Versicolor
- Virginica

The model will use four measurements:

- Sepal length
- Sepal width
- Petal length
- Petal width

This project follows the complete beginner deep-learning workflow:

**Load data → prepare features → build network → train → validate → evaluate → experiment**


## Learning goals

By the end of the project, you should be able to:

- Prepare data for a neural network.
- Create a simple neural network with **TensorFlow/Keras**.
- Identify the **input**, **hidden**, and **output** layers.
- Train a model using epochs and batch size.
- Use validation data to watch learning during training.
- Plot training and validation accuracy/loss.
- Evaluate predictions on unseen test data.
- Experiment with the number of neurons or layers.


## 1. Import the libraries

We use:

- `pandas` for viewing data.
- `matplotlib` for graphs.
- `scikit-learn` for the Iris dataset, splitting data, scaling, and evaluation.
- `TensorFlow/Keras` for the neural network.

> **Google Colab:** TensorFlow is normally already installed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Make results more repeatable when students rerun the notebook.
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


## 2. Load the Iris dataset

The Iris dataset is small, clean, and beginner-friendly.

Each row represents one flower.  
The **features** are measurements.  
The **label** is the flower species.


In [ ]:
iris = load_iris()

# X contains the four numeric input features.
X = iris.data

# y contains the species labels:
# 0 = setosa, 1 = versicolor, 2 = virginica
y = iris.target

# Put the data into a DataFrame so it is easy to inspect.
df = pd.DataFrame(X, columns=iris.feature_names)
df["species"] = [iris.target_names[label] for label in y]

print(df.head())
print("\nDataset shape:", df.shape)


### Quick exploration

Before training a model, always understand your data.

**Questions:**

1. How many flowers are in the dataset?
2. How many input features are there?
3. How many possible classes are there?


In [ ]:
print("Number of samples:", len(X))
print("Number of input features:", X.shape[1])
print("Classes:", iris.target_names)
print("\nClass counts:")
print(df["species"].value_counts())


## 3. Split the data into training, validation, and test sets

We use three groups:

- **Training data:** used to adjust the model's weights.
- **Validation data:** used to check learning while training.
- **Test data:** used only after training to measure performance on unseen data.

Here we use approximately:

- 60% training
- 20% validation
- 20% test


In [ ]:
# First, keep 20% of the data for the final test.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Then take 25% of the remaining 80% for validation.
# 25% of 80% = 20% of the full dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))


## 4. Scale the input features

Neural networks usually learn more easily when numeric features are on similar scales.

`StandardScaler` changes each feature so it is centered around `0` with a similar spread.

Important rule:

> Fit the scaler using the **training data only**. Then use the same scaler to transform validation and test data.


In [ ]:
scaler = StandardScaler()

# Learn scaling values from the training data.
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to validation and test data.
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("First scaled training example:")
print(X_train_scaled[0])


## 5. Build the neural network

Our network will contain:

### Input
Four flower measurements enter the network.

### Hidden layer
`8` neurons with the **ReLU** activation function.

### Output layer
`3` neurons because there are three flower species.

The output layer uses **softmax**, which creates three class probabilities that add up to about `1.0`.


In [ ]:
model = keras.Sequential([
    # The input shape is 4 because each flower has 4 features.
    keras.Input(shape=(4,)),

    # Hidden layer: 8 neurons using ReLU.
    layers.Dense(8, activation="relu"),

    # Output layer: 3 neurons, one for each flower class.
    layers.Dense(3, activation="softmax")
])

model.summary()


### What does the model summary show?

Look for:

- The number of layers.
- The output shape of each layer.
- The number of trainable parameters.

Those trainable parameters are mainly the **weights and biases** the model will learn.


## 6. Compile the model

We must tell Keras how the model should learn.

- **Optimizer: `adam`** — performs weight updates using gradient-based optimization.
- **Loss: `sparse_categorical_crossentropy`** — measures error for multi-class classification when labels are integers such as `0`, `1`, and `2`.
- **Accuracy** — tells us the percentage of correct classifications.


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 7. Train the neural network

Important terms:

- **Epoch:** one complete pass through the training data.
- **Batch size:** how many training examples are processed before one weight update.
- **Validation data:** unseen-by-training data used to monitor learning.

We will train for `60` epochs with a batch size of `8`.

Because the Iris dataset is tiny, training should be quick.


In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=8,
    verbose=0
)

print("Training complete.")
print("Final training accuracy:", round(history.history["accuracy"][-1], 3))
print("Final validation accuracy:", round(history.history["val_accuracy"][-1], 3))


## 8. Plot training and validation performance

These graphs help us understand whether the network is learning.

### Accuracy
Higher is better.

### Loss
Lower is better.

If training keeps improving while validation performance gets worse, the model may be **overfitting**.


In [ ]:
# Plot accuracy.
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs. Validation Accuracy")
plt.legend()
plt.show()


In [ ]:
# Plot loss.
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.legend()
plt.show()


### Interpret the graphs

Discuss with a partner or your instructor:

1. Does training accuracy generally increase?
2. Does training loss generally decrease?
3. Is validation accuracy close to training accuracy?
4. Do you see signs of overfitting?

> With a small dataset, the curves may bounce around. That is normal.


## 9. Evaluate the final model on the test set

The test set was not used for training or validation.

This gives us a final estimate of how well the model performs on unseen examples.


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("Test loss:", round(test_loss, 3))
print("Test accuracy:", round(test_accuracy, 3))


## 10. Make predictions

The model returns three probabilities for each flower.

Example:

`[0.02, 0.91, 0.07]`

means the model assigns:

- 2% probability to Setosa
- 91% probability to Versicolor
- 7% probability to Virginica

The largest probability becomes the predicted class.


In [ ]:
probabilities = model.predict(X_test_scaled, verbose=0)

# Convert probabilities into predicted class numbers.
predicted_classes = np.argmax(probabilities, axis=1)

# Show the first five predictions.
for i in range(5):
    print(
        f"Example {i + 1}:",
        "Predicted =", iris.target_names[predicted_classes[i]],
        "| Actual =", iris.target_names[y_test[i]],
        "| Confidence =", round(probabilities[i].max(), 3)
    )


## 11. Examine mistakes with a confusion matrix

A confusion matrix shows:

- Which classes were predicted correctly.
- Which classes were confused with one another.

This is more informative than accuracy alone.


In [ ]:
cm = confusion_matrix(y_test, predicted_classes)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)

display.plot()
plt.title("Iris Neural Network Confusion Matrix")
plt.show()


## 12. Student experiment: change the network

Now investigate how network design affects results.

Choose **one** experiment:

### Option A — More neurons
Change:

```python
layers.Dense(8, activation="relu")
```

to:

```python
layers.Dense(16, activation="relu")
```

### Option B — Add another hidden layer

Try:

```python
layers.Dense(8, activation="relu"),
layers.Dense(8, activation="relu"),
```

### Option C — Fewer neurons
Try only `4` hidden neurons.

Then retrain the model and compare the validation/test accuracy.


In [ ]:
# STUDENT EXPERIMENT
# Modify the architecture below, then run this cell.

experiment_model = keras.Sequential([
    keras.Input(shape=(4,)),

    # TODO: Change the number of neurons OR add another hidden layer.
    layers.Dense(8, activation="relu"),

    layers.Dense(3, activation="softmax")
])

experiment_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

experiment_history = experiment_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=8,
    verbose=0
)

experiment_loss, experiment_accuracy = experiment_model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("Original model test accuracy:", round(test_accuracy, 3))
print("Experiment model test accuracy:", round(experiment_accuracy, 3))


## 13. Project reflection

Write short answers:

1. What are the **four inputs** to this neural network?
2. What is the purpose of the **hidden layer**?
3. Why does the output layer contain **three neurons**?
4. What happens during an **epoch**?
5. What is the difference between **training data** and **validation data**?
6. Did your experimental network perform better, worse, or about the same?
7. Why should we **not** assume that a larger neural network is always better?

### Key takeaway

A neural network learns by repeatedly:

**making predictions → measuring error → adjusting weights and biases → trying again**

The Keras framework handles the difficult calculations, but you still need to understand the workflow and evaluate whether the model is learning well.
